In [8]:
pip install pandas torch transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 113.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

# AgentGraph — Trace Collector (Prototype v1)

A reusable **execution tracing pipeline** for LLM agents.

This notebook builds a lightweight, custom (no LangChain / no LangGraph) agent loop around a
local open-source Llama model. While the agent reasons, calls tools, and produces a final
response, **every execution event is recorded as a structured JSONL trace**.

> **Important:** This is *not* a prompt-injection detector. The collector only records *facts*.
> The labeling fields — `lep_injected`, `lep_type`, `lep_category`, `lep_location`,
> `lep_severity`, `risk_tags`, `caused_by_event`, `depends_on`, `propagates_to`,
> `downstream_failure`, `failure_type`, and `failure_event` — exist purely for *future*
> labeling/experiments and default to empty values unless manually specified.

Each event is one JSON object written as one line in an append-only `.jsonl` file. The schema is
deliberately generic so events can later become **graph nodes** and edges can be derived from
`event_id -> event_id+1`, `source -> target`, `tool_call -> tool_result`, `reasoning -> action`,
or — once labeled — `caused_by_event -> event`, `depends_on -> event`, and
`event -> propagates_to`, all without changing the schema. These causality edges are what let us
study how **Localized Execution Perturbations (LEPs)** propagate into downstream failures.

### Notebook layout
1. Imports
2. Configuration
3. Llama Model Loading
4. Tool Definitions
5. TraceCollector Implementation
6. Agent Implementation
7. Example Tasks
8. JSONL Generation
9. Visualization

## 1. Imports

Standard library + a small set of third-party dependencies. `transformers` / `torch` are only
*needed* if you want to run the real Llama model; the notebook degrades gracefully to a
deterministic mock LLM if they are unavailable, so it always runs top-to-bottom.

In [9]:
# --- Standard library ---
import os
import re
import json
import uuid
import textwrap
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

# --- Third party ---
import pandas as pd

# Optional ML stack. We import lazily/defensively so the notebook still runs end-to-end
# on a machine without torch/transformers (it will use the deterministic mock LLM instead).
try:
    import torch  # noqa: F401
    from transformers import AutoModelForCausalLM, AutoTokenizer

    _TRANSFORMERS_AVAILABLE = True
except Exception as _import_err:  # pragma: no cover - environment dependent
    _TRANSFORMERS_AVAILABLE = False
    _TRANSFORMERS_IMPORT_ERROR = _import_err

print("pandas:", pd.__version__)
print("transformers/torch available:", _TRANSFORMERS_AVAILABLE)

/home/cc/agentgraph-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas: 3.0.3
transformers/torch available: True


## 2. Configuration

All tunable knobs live here. `MODEL_NAME` points at the latest small open-source Llama
instruct model (3.2-1B) so it can run on modest hardware; swap it for any HF causal LM.
`USE_REAL_MODEL` lets you force the deterministic mock backend (handy for CI / offline runs).

In [10]:
# --- Model configuration ---
# Latest small open-source Llama instruct model. Any HF causal LM id works here.
MODEL_NAME: str = "meta-llama/Llama-3.2-1B-Instruct"

# Set False to always use the deterministic mock LLM (offline / reproducible runs).
# When True, we still fall back to the mock automatically if loading fails.
USE_REAL_MODEL: bool = True

# Generation settings (only used by the real backend).
MAX_NEW_TOKENS: int = 256
TEMPERATURE: float = 0.0  # deterministic-ish decoding for cleaner traces

# --- Agent configuration ---
MAX_AGENT_STEPS: int = 6  # safety cap on the reason -> act loop

# --- Trace I/O configuration ---
TRACE_DIR: Path = Path("traces")
TRACE_DIR.mkdir(exist_ok=True)

# Truncate long strings stored in *_summary fields so traces stay readable.
SUMMARY_MAX_CHARS: int = 280

print(f"MODEL_NAME      = {MODEL_NAME}")
print(f"USE_REAL_MODEL  = {USE_REAL_MODEL}")
print(f"TRACE_DIR       = {TRACE_DIR.resolve()}")

MODEL_NAME      = meta-llama/Llama-3.2-1B-Instruct
USE_REAL_MODEL  = True
TRACE_DIR       = /home/cc/traces


## 3. Llama Model Loading

We expose a tiny `LLMBackend` interface with a single job: given the current task + scratchpad,
return a structured decision `{reasoning, action, action_input, final_response}`.

- **`RealLlamaBackend`** prompts the loaded Llama model and parses a JSON action out of its
  output. If parsing fails it falls back to the deterministic planner.
- **`MockLlamaBackend`** skips the model entirely and uses the deterministic planner directly.

The deterministic planner (`plan_next_step`) is the transparent control logic shared by both
backends. It guarantees the notebook runs offline and that tools actually get exercised, while
keeping the *reasoning text* human-readable.

In [12]:
# A single agent "decision" returned by any backend.
@dataclass
class AgentDecision:
    """Structured output of one reasoning step.

    `action` is either a registered tool name or the sentinel "final".
    """

    reasoning: str
    action: str                      # tool name | "final"
    action_input: str = ""           # argument passed to the tool
    final_response: str = ""         # only populated when action == "final"


def plan_next_step(task: str, history: List[Dict[str, Any]], tool_names: List[str]) -> AgentDecision:
    """Deterministic planner — the transparent 'brain' shared by all backends.

    It inspects the task and the tools already used (``history``) and decides the next
    action. Each relevant tool is invoked at most once, after which the agent finalizes.
    This is intentionally simple and rule-based so traces are reproducible.
    """
    task_l = task.lower()
    used = {h["tool"] for h in history}

    # Map intent keywords -> tool name. Order defines execution priority.
    intents = [
        ("read_document", ["document", "read", "report", "file", "doc"]),
        ("search_notes", ["search", "find", "notes", "look up", "lookup", "note"]),
        ("calculator", ["calculate", "compute", "math", "sum", "multiply", "+", "*", "/", "average"]),
    ]

    for tool_name, keywords in intents:
        if tool_name not in tool_names:
            continue
        if tool_name in used:
            continue
        if any(k in task_l for k in keywords):
            return AgentDecision(
                reasoning=(
                    f"The task seems to require '{tool_name}'. I have not called it yet, "
                    f"so I will invoke it to gather the information I need."
                ),
                action=tool_name,
                action_input=task,
            )

    # Nothing left to do -> synthesize a final answer from observations.
    if history:
        observed = "; ".join(f"{h['tool']} -> {h['output']}" for h in history)
        final = f"Based on the tools I used ({observed}), here is my answer to: '{task}'."
    else:
        final = f"I can answer directly without tools: '{task}'."
    return AgentDecision(
        reasoning="I have gathered enough information; I will now compose the final response.",
        action="final",
        final_response=final,
    )

In [13]:
class LLMBackend:
    """Common interface. Subclasses must implement `decide`."""

    name: str = "base"

    def decide(self, task: str, history: List[Dict[str, Any]], tools: Dict[str, "Tool"]) -> AgentDecision:
        raise NotImplementedError


class MockLlamaBackend(LLMBackend):
    """Deterministic, offline backend. Delegates entirely to `plan_next_step`."""

    name = "mock-llama"

    def decide(self, task, history, tools):
        return plan_next_step(task, history, list(tools.keys()))


class RealLlamaBackend(LLMBackend):
    """Wraps a HuggingFace causal LM. Prompts for a JSON action and parses it.

    Any failure (load error, unparseable output, unknown action) degrades gracefully
    to the deterministic planner, so behaviour is always well-defined.
    """

    name = "real-llama"

    def __init__(self, model_name: str):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto",
        )
        self.model.eval()

    def _build_prompt(self, task, history, tools) -> str:
        tool_docs = "\n".join(f"- {n}: {t.description}" for n, t in tools.items())
        scratch = "\n".join(
            f"Step {i+1}: called {h['tool']}({h['input']!r}) -> {h['output']}"
            for i, h in enumerate(history)
        ) or "(no tools called yet)"
        system = (
            "You are a tool-using agent. Respond with ONLY a single JSON object, no prose.\n"
            "Schema: {\"reasoning\": str, \"action\": str, \"action_input\": str, "
            "\"final_response\": str}.\n"
            f"`action` must be one of: {list(tools.keys()) + ['final']}.\n"
            "Use 'final' (and fill final_response) when you can answer.\n\n"
            f"Available tools:\n{tool_docs}"
        )
        user = f"Task: {task}\n\nScratchpad:\n{scratch}\n\nReturn the JSON action now."
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _generate(self, prompt: str) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=TEMPERATURE > 0,
                temperature=max(TEMPERATURE, 1e-4),
                pad_token_id=self.tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    @staticmethod
    def _parse(raw: str, allowed: List[str]) -> Optional[AgentDecision]:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            return None
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            return None
        if data.get("action") not in allowed:
            return None
        return AgentDecision(
            reasoning=str(data.get("reasoning", "")),
            action=str(data["action"]),
            action_input=str(data.get("action_input", "")),
            final_response=str(data.get("final_response", "")),
        )

    def decide(self, task, history, tools):
        allowed = list(tools.keys()) + ["final"]
        try:
            raw = self._generate(self._build_prompt(task, history, tools))
            parsed = self._parse(raw, allowed)
            if parsed is not None:
                return parsed
        except Exception as err:  # pragma: no cover - runtime/model dependent
            print(f"[RealLlamaBackend] generation failed, using planner fallback: {err}")
        return plan_next_step(task, history, list(tools.keys()))


def load_llm_backend() -> LLMBackend:
    """Factory: try the real model, otherwise fall back to the mock backend."""
    if USE_REAL_MODEL and _TRANSFORMERS_AVAILABLE:
        try:
            print(f"Loading real model '{MODEL_NAME}' (first run downloads weights)...")
            backend = RealLlamaBackend(MODEL_NAME)
            print("Real Llama backend ready.")
            return backend
        except Exception as err:
            print(f"Could not load '{MODEL_NAME}': {err}\nFalling back to mock backend.")
    else:
        reason = "USE_REAL_MODEL=False" if not USE_REAL_MODEL else "transformers/torch unavailable"
        print(f"Using mock backend ({reason}).")
    return MockLlamaBackend()


# Instantiated after the Tool class is defined (see Section 4) so type hints resolve cleanly.
llm_backend: Optional[LLMBackend] = None

In [14]:
from huggingface_hub import notebook_login

notebook_login()

## 4. Tool Definitions

Three deterministic tools wrapped in a small `Tool` dataclass so the agent can discover them by
name. Determinism keeps the generated traces reproducible. Each tool returns a structured
`dict` with `output` (the human-readable result, recorded in `output_summary`) and optional
`metadata` (tool-internal detail the agent may use; it is **not** persisted in the trace schema).

In [15]:
@dataclass
class Tool:
    """A callable tool the agent can invoke by name."""

    name: str
    description: str
    func: Callable[[str], Dict[str, Any]]

    def __call__(self, tool_input: str) -> Dict[str, Any]:
        return self.func(tool_input)


# --- In-memory deterministic "knowledge" the tools draw from ---
_DOCUMENTS: Dict[str, str] = {
    "q3_report": "Q3 revenue was 1200 units at 35 USD each. Operating costs were 18000 USD.",
    "onboarding": "Welcome to AgentGraph. Step 1: install deps. Step 2: run the demo notebook.",
    "default": "This is a placeholder document with no special content.",
}

_NOTES: List[str] = [
    "Meeting note: ship the trace collector prototype by Friday.",
    "Idea: derive graph edges from event_id -> event_id+1.",
    "Reminder: traces must stay append-only and schema-stable.",
    "Bug: remember to truncate long summaries to keep JSONL readable.",
]


def read_document(tool_input: str) -> Dict[str, Any]:
    """Return the contents of a named document (deterministic lookup)."""
    key = tool_input.strip().lower()
    doc_id = next((k for k in _DOCUMENTS if k in key), "default")
    content = _DOCUMENTS[doc_id]
    return {"output": content, "metadata": {"doc_id": doc_id, "char_len": len(content)}}


def search_notes(tool_input: str) -> Dict[str, Any]:
    """Return notes containing any query token (deterministic substring search)."""
    query = tool_input.strip().lower()
    tokens = [t for t in re.split(r"\W+", query) if len(t) > 2]
    hits = [n for n in _NOTES if any(t in n.lower() for t in tokens)] or _NOTES[:2]
    return {"output": " | ".join(hits), "metadata": {"query": query, "num_hits": len(hits)}}


def calculator(tool_input: str) -> Dict[str, Any]:
    """Safely evaluate a basic arithmetic expression extracted from the input.

    We isolate *contiguous* arithmetic runs (so stray digits inside words like
    'q3_report' don't get merged into the expression) and pick the longest run that
    actually contains an operator.
    """
    runs = re.findall(r"[0-9.\+\-\*/()\s]+", tool_input)
    candidates = [r.strip() for r in runs if re.search(r"[-+*/]", r) and re.search(r"\d", r)]
    expr = max(candidates, key=len) if candidates else ""
    try:
        # Restricted eval: no builtins, arithmetic characters only (validated above).
        result = eval(expr, {"__builtins__": {}}, {}) if expr else "no expression found"
    except Exception as err:
        result = f"error: {err}"
    return {"output": str(result), "metadata": {"expression": expr}}


# Registry the agent will use to look tools up by name.
TOOLS: Dict[str, Tool] = {
    "read_document": Tool(
        "read_document", "Read a named document by keyword (e.g. 'q3_report').", read_document
    ),
    "search_notes": Tool(
        "search_notes", "Search internal notes for a query string.", search_notes
    ),
    "calculator": Tool(
        "calculator", "Evaluate a basic arithmetic expression.", calculator
    ),
}

# Quick sanity check.
for _name, _tool in TOOLS.items():
    print(f"{_name:14s} -> {_tool('q3 report 1200 * 35')['output'][:60]}")

read_document  -> This is a placeholder document with no special content.
search_notes   -> Meeting note: ship the trace collector prototype by Friday. 
calculator     -> 42000


In [16]:
# Now that tools exist, instantiate the backend (real model if available, else mock).
llm_backend = load_llm_backend()
print("Active backend:", llm_backend.name)

Loading real model 'meta-llama/Llama-3.2-1B-Instruct' (first run downloads weights)...
Could not load 'meta-llama/Llama-3.2-1B-Instruct': Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`
Falling back to mock backend.
Active backend: mock-llama


## 5. TraceCollector Implementation

The heart of the project. `TraceEvent` is a dataclass mirroring the exact JSONL schema. The
labeling fields fall into four groups, all of which default to empty/`None` and are **never**
set by the collector itself — only later labeling may set them:

- **LEP labeling**: `lep_injected`, `lep_type`, `lep_category`, `lep_location`, `lep_severity`
- **forensic**: `risk_tags`
- **causality / dependency**: `caused_by_event`, `depends_on`, `propagates_to`
- **failure outcome**: `downstream_failure`, `failure_type`, `failure_event`

The factual `expected_behavior` / `observed_behavior` fields may be filled at log time.

`TraceCollector` manages a single append-only `.jsonl` file:

- `start_trace()` — begin a new trace (fresh `trace_id`, reset event counter, open a file).
- `log_event(...)` — build one event and **immediately append** it to disk as one line.
- `save_jsonl()` — (re)write the full in-memory trace to disk in one shot.
- `load_jsonl()` — read a `.jsonl` file back into a list of event dicts.
- `pretty_print_trace()` — human-readable console rendering.

In [17]:
# The set of valid event types (kept as a constant for documentation + light validation).
EVENT_TYPES = (
    "user_input",
    "reasoning",
    "tool_call",
    "tool_result",
    "llm_output",
    "final_response",
)


def _utc_now_iso() -> str:
    """ISO-8601 UTC timestamp (sortable, timezone-aware)."""
    return datetime.now(timezone.utc).isoformat()


def _summarize(value: Any, limit: int = SUMMARY_MAX_CHARS) -> str:
    """Coerce a value to a trimmed single-line string for *_summary fields."""
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    text = " ".join(text.split())  # collapse whitespace/newlines
    return text if len(text) <= limit else text[: limit - 1] + "\u2026"


@dataclass
class TraceEvent:
    """One execution event == one JSON line. Mirrors the required JSONL schema exactly.

    Beyond the raw facts (source/target/summaries), the schema carries fields designed to
    support *future* graph analysis of Localized Execution Perturbations (LEPs) and how they
    propagate into downstream failures:

    - behavior (`expected_behavior`, `observed_behavior`): factual descriptions; may be filled
      by the agent at log time.
    - LEP labeling (`lep_injected`, `lep_type`, `lep_category`, `lep_location`, `lep_severity`):
      set ONLY during later labeling/injection experiments — never by the collector.
    - causality / dependency (`caused_by_event`, `depends_on`, `propagates_to`): the edges that
      let us reconstruct a propagation graph; also labeling-only.
    - failure outcome (`downstream_failure`, `failure_type`, `failure_event`): whether/how this
      event leads to a failure, and which event id the failure manifests at; labeling-only.
    """

    trace_id: str
    event_id: int
    timestamp: str
    event_type: str
    source: str
    target: str
    input_summary: str
    output_summary: str

    # --- Behavior description (factual; may be filled by the agent at log time) ---
    expected_behavior: str = ""
    observed_behavior: str = ""

    # --- LEP labeling: for FUTURE labeling/injection only; collector never sets these. ---
    lep_injected: bool = False            # was a perturbation deliberately injected here?
    lep_type: Optional[str] = None        # e.g. "1.3 Step Repetition"
    lep_category: Optional[str] = None    # e.g. "FC1"
    lep_location: Optional[str] = None    # where the perturbation manifests
    lep_severity: Optional[str] = None    # e.g. "low" | "medium" | "high"

    # --- Forensic ---
    risk_tags: List[str] = field(default_factory=list)

    # --- Causality / dependency: edges of the LEP propagation graph (labeling-only). ---
    caused_by_event: Optional[int] = None                    # event_id that caused this one
    depends_on: List[int] = field(default_factory=list)      # event_ids this one depends on
    propagates_to: List[int] = field(default_factory=list)   # event_ids this propagates into

    # --- Failure outcome (labeling-only). ---
    downstream_failure: bool = False          # did this lead to a downstream failure?
    failure_type: Optional[str] = None        # e.g. "non_termination" | "wrong_answer"
    failure_event: Optional[int] = None       # event_id where the failure manifests

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class TraceCollector:
    """Records agent execution as an append-only JSONL trace.

    The collector only records *facts*. It makes no judgement about whether a trace is
    normal or abnormal.
    """

    def __init__(self, trace_dir: Path = TRACE_DIR):
        self.trace_dir = Path(trace_dir)
        self.trace_dir.mkdir(exist_ok=True)
        self.trace_id: Optional[str] = None
        self.path: Optional[Path] = None
        self._next_event_id: int = 1
        self.events: List[TraceEvent] = []

    # --- lifecycle ---------------------------------------------------------
    def start_trace(self, trace_id: Optional[str] = None) -> str:
        """Begin a new trace; returns the trace_id and opens a fresh JSONL file."""
        self.trace_id = trace_id or uuid.uuid4().hex[:12]
        self._next_event_id = 1
        self.events = []
        self.path = self.trace_dir / f"trace_{self.trace_id}.jsonl"
        # Truncate/create the file so each trace starts clean.
        self.path.write_text("", encoding="utf-8")
        return self.trace_id

    # --- writing -----------------------------------------------------------
    def log_event(
        self,
        event_type: str,
        source: str,
        target: str,
        input_summary: Any = "",
        output_summary: Any = "",
        *,
        expected_behavior: str = "",
        observed_behavior: str = "",
        lep_injected: bool = False,
        lep_type: Optional[str] = None,
        lep_category: Optional[str] = None,
        lep_location: Optional[str] = None,
        lep_severity: Optional[str] = None,
        risk_tags: Optional[List[str]] = None,
        caused_by_event: Optional[int] = None,
        depends_on: Optional[List[int]] = None,
        propagates_to: Optional[List[int]] = None,
        downstream_failure: bool = False,
        failure_type: Optional[str] = None,
        failure_event: Optional[int] = None,
    ) -> TraceEvent:
        """Create one event and immediately append it to disk as a single JSONL line.

        Only `expected_behavior` / `observed_behavior` are expected to be set during normal
        runs. All LEP / causality / failure fields default to empty and are intended to be
        filled in later by labeling tools (see `annotate_lep` / `link_propagation`).
        """
        if self.trace_id is None or self.path is None:
            raise RuntimeError("Call start_trace() before log_event().")
        if event_type not in EVENT_TYPES:
            raise ValueError(f"Unknown event_type {event_type!r}; expected one of {EVENT_TYPES}")

        event = TraceEvent(
            trace_id=self.trace_id,
            event_id=self._next_event_id,
            timestamp=_utc_now_iso(),
            event_type=event_type,
            source=source,
            target=target,
            input_summary=_summarize(input_summary),
            output_summary=_summarize(output_summary),
            expected_behavior=_summarize(expected_behavior),
            observed_behavior=_summarize(observed_behavior),
            lep_injected=lep_injected,
            lep_type=lep_type,
            lep_category=lep_category,
            lep_location=lep_location,
            lep_severity=lep_severity,
            risk_tags=list(risk_tags) if risk_tags else [],
            caused_by_event=caused_by_event,
            depends_on=list(depends_on) if depends_on else [],
            propagates_to=list(propagates_to) if propagates_to else [],
            downstream_failure=downstream_failure,
            failure_type=failure_type,
            failure_event=failure_event,
        )
        self._next_event_id += 1
        self.events.append(event)

        # Append-only write: one JSON object per line, flushed immediately.
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(event.to_dict(), default=str) + "\n")
        return event

    # --- persistence -------------------------------------------------------
    def save_jsonl(self, path: Optional[Path] = None) -> Path:
        """Rewrite the full in-memory trace to disk in one shot (idempotent snapshot)."""
        target = Path(path) if path else self.path
        if target is None:
            raise RuntimeError("No path to save to; start a trace first.")
        with target.open("w", encoding="utf-8") as fh:
            for event in self.events:
                fh.write(json.dumps(event.to_dict(), default=str) + "\n")
        return target

    @staticmethod
    def load_jsonl(path: Path) -> List[Dict[str, Any]]:
        """Load a JSONL trace file into a list of event dicts."""
        rows: List[Dict[str, Any]] = []
        with Path(path).open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    # --- display -----------------------------------------------------------
    def pretty_print_trace(self) -> None:
        """Render the current trace to the console in a readable, ordered format."""
        print(f"=== Trace {self.trace_id} ({len(self.events)} events) ===")
        for ev in self.events:
            print(
                f"[{ev.event_id:>2}] {ev.event_type:<14} "
                f"{ev.source} -> {ev.target}"
            )
            if ev.input_summary:
                print(f"      in : {ev.input_summary}")
            if ev.output_summary:
                print(f"      out: {ev.output_summary}")


print("TraceCollector ready. Event types:", EVENT_TYPES)

TraceCollector ready. Event types: ('user_input', 'reasoning', 'tool_call', 'tool_result', 'llm_output', 'final_response')


### 5b. LEP Taxonomy & Labeling Helpers

The **Localized Execution Perturbation (LEP)** taxonomy groups failure modes into three
failure categories (FC1–FC3). The collector never assigns these — they are applied *after the
fact* by labeling tools / human review. Two helpers make labeling explicit and consistent:

- `annotate_lep(events, event_id, lep_code, ...)` — tag a single event with an LEP (the
  `lep_category` is auto-derived from the taxonomy); optionally set `lep_severity`,
  `lep_injected`, and failure fields (`downstream_failure`, `failure_type`, `failure_event`).
- `link_propagation(events, cause_id, effect_ids)` — record causal edges: the cause event's
  `propagates_to` gains the effect ids, and each effect's `caused_by_event` / `depends_on`
  point back to the cause.

Both operate on a list of loaded event dicts so a labeled trace can be re-saved and later turned
into a propagation graph.

In [18]:
# --- Localized Execution Perturbation (LEP) taxonomy -------------------------
# Three failure categories, each mapping LEP code -> human-readable definition.
LEP_TAXONOMY: Dict[str, Dict[str, Any]] = {
    "FC1": {
        "name": "System Design Issues",
        "description": "Failures originating from system design decisions, prompt "
                       "specifications, or state management.",
        "leps": {
            "1.1": "Disobey Task Specification",
            "1.2": "Disobey Role Specification",
            "1.3": "Step Repetition",
            "1.4": "Loss of Conversation History",
            "1.5": "Unaware of Termination Conditions",
        },
    },
    "FC2": {
        "name": "Inter-Agent Misalignment",
        "description": "Failures arising from breakdowns in information flow and coordination "
                       "between agents.",
        "leps": {
            "2.1": "Conversation Reset",
            "2.2": "Fail to Ask for Clarification",
            "2.3": "Task Derailment",
            "2.4": "Information Withholding",
            "2.5": "Ignored Other Agent's Input",
            "2.6": "Reasoning-Action Mismatch",
        },
    },
    "FC3": {
        "name": "Task Verification",
        "description": "Failures involving inadequate verification or quality control.",
        "leps": {
            "3.1": "Premature Termination",
            "3.2": "No or Incomplete Verification",
            "3.3": "Incorrect Verification",
        },
    },
}

# Flat lookups: "1.3" -> "FC1" and "1.3" -> "Step Repetition".
LEP_CODE_TO_CATEGORY: Dict[str, str] = {
    code: cat for cat, spec in LEP_TAXONOMY.items() for code in spec["leps"]
}
LEP_CODE_TO_NAME: Dict[str, str] = {
    code: name for spec in LEP_TAXONOMY.values() for code, name in spec["leps"].items()
}


def _find_event(events: List[Dict[str, Any]], event_id: int) -> Dict[str, Any]:
    for ev in events:
        if ev["event_id"] == event_id:
            return ev
    raise KeyError(f"event_id {event_id} not found in trace")


def annotate_lep(
    events: List[Dict[str, Any]],
    event_id: int,
    lep_code: str,
    *,
    lep_location: Optional[str] = None,
    lep_severity: Optional[str] = None,
    lep_injected: bool = False,
    downstream_failure: bool = False,
    failure_type: Optional[str] = None,
    failure_event: Optional[int] = None,
) -> Dict[str, Any]:
    """Label one event with an LEP. `lep_category` is auto-derived from the taxonomy.

    `lep_code` is a taxonomy code such as "1.3". The stored `lep_type` is "1.3 Step Repetition".
    `lep_injected` marks deliberately injected perturbations; `lep_severity` is a free-form
    severity ("low"/"medium"/"high"). When the event culminates in a failure, set
    `downstream_failure` plus `failure_type` / `failure_event`. Returns the mutated event dict.
    """
    if lep_code not in LEP_CODE_TO_CATEGORY:
        raise ValueError(f"Unknown LEP code {lep_code!r}; valid codes: {list(LEP_CODE_TO_CATEGORY)}")
    ev = _find_event(events, event_id)
    ev["lep_injected"] = lep_injected
    ev["lep_type"] = f"{lep_code} {LEP_CODE_TO_NAME[lep_code]}"
    ev["lep_category"] = LEP_CODE_TO_CATEGORY[lep_code]
    ev["lep_location"] = lep_location if lep_location is not None else ev.get("event_type")
    ev["lep_severity"] = lep_severity
    ev["downstream_failure"] = downstream_failure
    ev["failure_type"] = failure_type
    ev["failure_event"] = failure_event
    return ev


def link_propagation(
    events: List[Dict[str, Any]],
    cause_id: int,
    effect_ids: List[int],
) -> None:
    """Record causal propagation: cause -> effects.

    Adds each effect id to the cause's `propagates_to`, sets each effect's `caused_by_event`
    to the cause id, and records the reverse `depends_on` edge on each effect — together
    forming the directed edges of the propagation graph.
    """
    cause = _find_event(events, cause_id)
    cause.setdefault("propagates_to", [])
    for eid in effect_ids:
        effect = _find_event(events, eid)
        if eid not in cause["propagates_to"]:
            cause["propagates_to"].append(eid)
        effect["caused_by_event"] = cause_id
        effect.setdefault("depends_on", [])
        if cause_id not in effect["depends_on"]:
            effect["depends_on"].append(cause_id)


print("LEP taxonomy loaded:",
      {cat: len(spec["leps"]) for cat, spec in LEP_TAXONOMY.items()})

LEP taxonomy loaded: {'FC1': 5, 'FC2': 6, 'FC3': 3}


## 6. Agent Implementation

A minimal, transparent loop (no LangChain / LangGraph):

```
user_input
   ↓
reasoning (LLM)  ←────────────┐
   ↓                          │
 tool_call → tool_result ─────┘  (repeat while the agent picks a tool)
   ↓
llm_output → final_response
```

Every step emits trace events automatically via the `TraceCollector`. The agent itself is
oblivious to the forensic fields — it always logs facts with the safe defaults.

In [19]:
class Agent:
    """A lightweight reason -> act -> observe loop that records a full trace."""

    def __init__(
        self,
        llm: LLMBackend,
        tools: Dict[str, Tool],
        collector: TraceCollector,
        max_steps: int = MAX_AGENT_STEPS,
    ):
        self.llm = llm
        self.tools = tools
        self.collector = collector
        self.max_steps = max_steps

    def run(self, task: str) -> Dict[str, Any]:
        """Execute one task end-to-end, emitting trace events at every step."""
        trace_id = self.collector.start_trace()

        # 1) The user hands a task to the agent.
        self.collector.log_event(
            event_type="user_input",
            source="user",
            target="agent",
            input_summary=task,
            output_summary="",
            expected_behavior="User submits a well-formed task for the agent to solve.",
            observed_behavior=f"User submitted task: {task}",
        )

        history: List[Dict[str, Any]] = []  # observations fed back into the LLM
        final_response = ""

        for step in range(1, self.max_steps + 1):
            # 2) LLM reasons about the next step.
            decision = self.llm.decide(task, history, self.tools)
            self.collector.log_event(
                event_type="reasoning",
                source="llm",
                target="internal",
                input_summary=task,
                output_summary=decision.reasoning,
                expected_behavior="Agent reasons toward the task goal and picks a valid next action.",
                observed_behavior=f"Agent reasoned and chose action '{decision.action}'.",
            )

            # 3) Terminal branch: the LLM decided to answer.
            if decision.action == "final":
                final_response = decision.final_response or "(no answer produced)"
                # llm_output captures the raw model decision to finalize...
                self.collector.log_event(
                    event_type="llm_output",
                    source="llm",
                    target="agent",
                    input_summary=task,
                    output_summary=final_response,
                    expected_behavior="Agent produces a final answer grounded in the gathered observations.",
                    observed_behavior="Agent emitted a final answer.",
                )
                # ...and final_response is the answer handed back to the user.
                self.collector.log_event(
                    event_type="final_response",
                    source="agent",
                    target="user",
                    input_summary=task,
                    output_summary=final_response,
                    expected_behavior="Agent returns a correct, complete answer to the user.",
                    observed_behavior="Agent returned the final answer to the user.",
                )
                break

            # 4) Tool branch: validate, log the call, execute, log the result.
            tool = self.tools.get(decision.action)
            if tool is None:
                # Unknown action -> record it as reasoning and stop to stay safe.
                self.collector.log_event(
                    event_type="reasoning",
                    source="llm",
                    target="internal",
                    input_summary=decision.action,
                    output_summary=f"Unknown action '{decision.action}'; aborting loop.",
                    expected_behavior="Agent selects a registered tool or 'final'.",
                    observed_behavior=f"Agent selected unknown action '{decision.action}'.",
                )
                final_response = f"Could not complete task: unknown action {decision.action!r}."
                self.collector.log_event(
                    event_type="final_response", source="agent", target="user",
                    input_summary=task, output_summary=final_response,
                    expected_behavior="Agent returns a correct, complete answer to the user.",
                    observed_behavior="Agent aborted due to an unknown action.",
                )
                break

            self.collector.log_event(
                event_type="tool_call",
                source="agent",
                target=tool.name,
                input_summary=decision.action_input,
                output_summary="",
                expected_behavior=f"Agent invokes '{tool.name}' with an input appropriate for the task.",
                observed_behavior=f"Agent called '{tool.name}' with input: {decision.action_input}",
            )

            result = tool(decision.action_input)
            self.collector.log_event(
                event_type="tool_result",
                source=tool.name,
                target="agent",
                input_summary=decision.action_input,
                output_summary=result.get("output", ""),
                expected_behavior=f"'{tool.name}' returns a correct, usable result.",
                observed_behavior=f"'{tool.name}' returned: {result.get('output', '')}",
            )

            history.append({
                "tool": tool.name,
                "input": decision.action_input,
                "output": result.get("output", ""),
            })
        else:
            # Loop exhausted without finalizing -> emit a fallback final_response.
            final_response = "Reached max steps without a final answer."
            self.collector.log_event(
                event_type="final_response",
                source="agent",
                target="user",
                input_summary=task,
                output_summary=final_response,
                expected_behavior="Agent recognizes task completion and returns the answer within the step budget.",
                observed_behavior="Agent exhausted its step budget without producing a final answer.",
            )

        return {
            "trace_id": trace_id,
            "final_response": final_response,
            "num_events": len(self.collector.events),
            "path": str(self.collector.path),
        }


print("Agent ready.")

Agent ready.


## 7. Example Tasks

A few tasks chosen to exercise each tool and combinations of tools. The deterministic planner
routes each task to the relevant tool(s), so these produce reproducible multi-event traces.

In [20]:
EXAMPLE_TASKS: List[str] = [
    # Exercises read_document + calculator (read the q3 report, then compute revenue).
    "Read the q3_report document and calculate 1200 * 35 for total revenue.",
    # Exercises search_notes only.
    "Search notes for anything about graph edges.",
    # Exercises calculator only.
    "Calculate (18000 / 1200) + 5 for the unit break-even.",
]

for i, t in enumerate(EXAMPLE_TASKS, 1):
    print(f"{i}. {t}")

1. Read the q3_report document and calculate 1200 * 35 for total revenue.
2. Search notes for anything about graph edges.
3. Calculate (18000 / 1200) + 5 for the unit break-even.


## 8. JSONL Generation

Run the agent over the example tasks. Each run writes its own append-only `.jsonl` file under
`traces/`. We keep the first run's path around for the visualization section.

In [21]:
collector = TraceCollector()
agent = Agent(llm=llm_backend, tools=TOOLS, collector=collector)

run_results: List[Dict[str, Any]] = []
for task in EXAMPLE_TASKS:
    result = agent.run(task)
    run_results.append(result)
    print(f"task: {task}")
    print(f"  -> trace_id={result['trace_id']}  events={result['num_events']}  file={result['path']}")
    print(f"  -> final: {result['final_response']}\n")

# Keep the first trace file for the visualization section.
PRIMARY_TRACE_PATH = Path(run_results[0]["path"])
print("Primary trace file:", PRIMARY_TRACE_PATH.resolve())

task: Read the q3_report document and calculate 1200 * 35 for total revenue.
  -> trace_id=b864f75f5b08  events=10  file=traces/trace_b864f75f5b08.jsonl
  -> final: Based on the tools I used (read_document -> Q3 revenue was 1200 units at 35 USD each. Operating costs were 18000 USD.; calculator -> 42000), here is my answer to: 'Read the q3_report document and calculate 1200 * 35 for total revenue.'.

task: Search notes for anything about graph edges.
  -> trace_id=0022348687de  events=7  file=traces/trace_0022348687de.jsonl
  -> final: Based on the tools I used (search_notes -> Idea: derive graph edges from event_id -> event_id+1.), here is my answer to: 'Search notes for anything about graph edges.'.

task: Calculate (18000 / 1200) + 5 for the unit break-even.
  -> trace_id=f02ea2b882a7  events=7  file=traces/trace_f02ea2b882a7.jsonl
  -> final: Based on the tools I used (calculator -> 20.0), here is my answer to: 'Calculate (18000 / 1200) + 5 for the unit break-even.'.

Primary trace 

In [22]:
# Pretty-print the most recent trace, and show the raw JSONL lines of the primary trace.
collector.pretty_print_trace()

print("\n--- Raw JSONL (primary trace) ---")
print(PRIMARY_TRACE_PATH.read_text(encoding="utf-8").strip())

=== Trace f02ea2b882a7 (7 events) ===
[ 1] user_input     user -> agent
      in : Calculate (18000 / 1200) + 5 for the unit break-even.
[ 2] reasoning      llm -> internal
      in : Calculate (18000 / 1200) + 5 for the unit break-even.
      out: The task seems to require 'calculator'. I have not called it yet, so I will invoke it to gather the information I need.
[ 3] tool_call      agent -> calculator
      in : Calculate (18000 / 1200) + 5 for the unit break-even.
[ 4] tool_result    calculator -> agent
      in : Calculate (18000 / 1200) + 5 for the unit break-even.
      out: 20.0
[ 5] reasoning      llm -> internal
      in : Calculate (18000 / 1200) + 5 for the unit break-even.
      out: I have gathered enough information; I will now compose the final response.
[ 6] llm_output     llm -> agent
      in : Calculate (18000 / 1200) + 5 for the unit break-even.
      out: Based on the tools I used (calculator -> 20.0), here is my answer to: 'Calculate (18000 / 1200) + 5 for the u

## 9. Visualization

Load the JSONL back from disk and inspect it:

- a pandas DataFrame of all events,
- a printed event timeline,
- event-type counts,
- and a few summary statistics.

We also sketch how edges become a graph in a future step — without touching the schema.

In [23]:
# Load the JSONL from disk (round-trip through TraceCollector.load_jsonl).
events = TraceCollector.load_jsonl(PRIMARY_TRACE_PATH)
df = pd.DataFrame(events)

# Display core columns as a DataFrame.
display_cols = [
    "event_id", "event_type", "source", "target",
    "input_summary", "output_summary",
    "expected_behavior", "observed_behavior",
    "lep_injected", "lep_type", "lep_category", "lep_severity",
    "caused_by_event", "depends_on", "propagates_to",
    "downstream_failure", "failure_type", "failure_event",
]
df[display_cols]

,event_id,event_type,source,target,input_summary,output_summary,expected_behavior,observed_behavior,lep_injected,lep_type,lep_category,lep_severity,caused_by_event,depends_on,propagates_to,downstream_failure,failure_type,failure_event
0,1,user_input,user,agent,Read the q3_report document and calculate 1200...,,User submits a well-formed task for the agent ...,User submitted task: Read the q3_report docume...,False,None,None,None,None,[],[],False,None,None
1,2,reasoning,llm,internal,Read the q3_report document and calculate 1200...,The task seems to require 'read_document'. I h...,Agent reasons toward the task goal and picks a...,Agent reasoned and chose action 'read_document'.,False,None,None,None,None,[],[],False,None,None
2,3,tool_call,agent,read_document,Read the q3_report document and calculate 1200...,,Agent invokes 'read_document' with an input ap...,Agent called 'read_document' with input: Read ...,False,None,None,None,None,[],[],False,None,None
3,4,tool_result,read_document,agent,Read the q3_report document and calculate 1200...,Q3 revenue was 1200 units at 35 USD each. Oper...,"'read_document' returns a correct, usable result.",'read_document' returned: Q3 revenue was 1200 ...,False,None,None,None,None,[],[],False,None,None
4,5,reasoning,llm,internal,Read the q3_report document and calculate 1200...,The task seems to require 'calculator'. I have...,Agent reasons toward the task goal and picks a...,Agent reasoned and chose action 'calculator'.,False,None,None,None,None,[],[],False,None,None
5,6,tool_call,agent,calculator,Read the q3_report document and calculate 1200...,,Agent invokes 'calculator' with an input appro...,Agent called 'calculator' with input: Read the...,False,None,None,None,None,[],[],False,None,None
6,7,tool_result,calculator,agent,Read the q3_report document and calculate 1200...,42000,"'calculator' returns a correct, usable result.",'calculator' returned: 42000,False,None,None,None,None,[],[],False,None,None
7,8,reasoning,llm,internal,Read the q3_report document and calculate 1200...,I have gathered enough information; I will now...,Agent reasons toward the task goal and picks a...,Agent reasoned and chose action 'final'.,False,None,None,None,None,[],[],False,None,None
8,9,llm_output,llm,agent,Read the q3_report document and calculate 1200...,Based on the tools I used (read_document -> Q3...,Agent produces a final answer grounded in the ...,Agent emitted a final answer.,False,None,None,None,None,[],[],False,None,None
9,10,final_response,agent,user,Read the q3_report document and calculate 1200...,Based on the tools I used (read_document -> Q3...,"Agent returns a correct, complete answer to th...",Agent returned the final answer to the user.,False,None,None,None,None,[],[],False,None,None


In [24]:
# --- Event timeline ---
print(f"Event timeline for trace {df['trace_id'].iloc[0]}:\n")
for _, row in df.iterrows():
    arrow = f"{row['source']} -> {row['target']}"
    print(f"  #{row['event_id']:<2} {row['event_type']:<14} {arrow:<28} | {row['output_summary'][:60]}")

# --- Event counts ---
print("\nEvent counts by type:")
print(df["event_type"].value_counts().to_string())

# --- Trace statistics ---
ts = pd.to_datetime(df["timestamp"])
duration_s = (ts.max() - ts.min()).total_seconds()
print("\nTrace statistics:")
print(f"  trace_id        : {df['trace_id'].iloc[0]}")
print(f"  total events    : {len(df)}")
print(f"  tool calls      : {(df['event_type'] == 'tool_call').sum()}")
print(f"  distinct sources: {df['source'].nunique()}")
print(f"  distinct targets: {df['target'].nunique()}")
print(f"  span (seconds)  : {duration_s:.4f}")
print(f"  any risk_tags   : {df['risk_tags'].apply(bool).any()}  (expected False before labeling)")
print(f"  any lep_injected: {df['lep_injected'].any()}  (expected False before labeling)")
print(f"  any lep labeled : {df['lep_type'].notna().any()}  (expected False before labeling)")
print(f"  downstream fails: {int(df['downstream_failure'].sum())}  (expected 0 before labeling)")

Event timeline for trace b864f75f5b08:

  #1  user_input     user -> agent                | 
  #2  reasoning      llm -> internal              | The task seems to require 'read_document'. I have not called
  #3  tool_call      agent -> read_document       | 
  #4  tool_result    read_document -> agent       | Q3 revenue was 1200 units at 35 USD each. Operating costs we
  #5  reasoning      llm -> internal              | The task seems to require 'calculator'. I have not called it
  #6  tool_call      agent -> calculator          | 
  #7  tool_result    calculator -> agent          | 42000
  #8  reasoning      llm -> internal              | I have gathered enough information; I will now compose the f
  #9  llm_output     llm -> agent                 | Based on the tools I used (read_document -> Q3 revenue was 1
  #10 final_response agent -> user                | Based on the tools I used (read_document -> Q3 revenue was 1

Event counts by type:
event_type
reasoning         3
tool_call  

In [25]:
# --- Future compatibility: derive graph edges WITHOUT changing the schema ---
# This is just a demonstration of how events become nodes and edges later.

def derive_sequential_edges(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """event_id -> event_id+1 (temporal flow edges)."""
    return [
        {"from": rows[i]["event_id"], "to": rows[i + 1]["event_id"], "kind": "sequence"}
        for i in range(len(rows) - 1)
    ]


def derive_call_result_edges(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """tool_call -> the immediately following tool_result (causal edges)."""
    edges = []
    for i in range(len(rows) - 1):
        if rows[i]["event_type"] == "tool_call" and rows[i + 1]["event_type"] == "tool_result":
            edges.append({"from": rows[i]["event_id"], "to": rows[i + 1]["event_id"], "kind": "tool_call->result"})
    return edges


def derive_propagation_edges(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """LEP propagation / dependency edges derived from the labeling fields.

    Uses `propagates_to` (cause -> effect), `caused_by_event` (effect -> cause), and
    `depends_on` (dependency -> dependent). These only exist once a trace has been labeled
    (see `annotate_lep` / `link_propagation`).
    """
    edges = []
    for row in rows:
        src = row["event_id"]
        for dst in row.get("propagates_to") or []:
            edges.append({"from": src, "to": dst, "kind": "propagates_to"})
        cause = row.get("caused_by_event")
        if cause is not None:
            edges.append({"from": cause, "to": src, "kind": "caused_by"})
        for dep in row.get("depends_on") or []:
            edges.append({"from": dep, "to": src, "kind": "depends_on"})
    return edges


seq_edges = derive_sequential_edges(events)
call_edges = derive_call_result_edges(events)
prop_edges = derive_propagation_edges(events)

print(f"Nodes (events)         : {len(events)}")
print(f"Sequential edges       : {len(seq_edges)}")
print(f"tool_call->result edges: {len(call_edges)}")
print(f"LEP propagation edges  : {len(prop_edges)}")
print("\nExample sequential edges:")
for e in seq_edges[:4]:
    print(" ", e)
print("\ntool_call->result edges:")
for e in call_edges:
    print(" ", e)

print("\nDone. Each event is a future graph node; edges above are derived purely from the schema. "
      "Run the LEP-labeling demo below to populate propagation edges.")

Nodes (events)         : 10
Sequential edges       : 9
tool_call->result edges: 2
LEP propagation edges  : 0

Example sequential edges:
  {'from': 1, 'to': 2, 'kind': 'sequence'}
  {'from': 2, 'to': 3, 'kind': 'sequence'}
  {'from': 3, 'to': 4, 'kind': 'sequence'}
  {'from': 4, 'to': 5, 'kind': 'sequence'}

tool_call->result edges:
  {'from': 3, 'to': 4, 'kind': 'tool_call->result'}
  {'from': 6, 'to': 7, 'kind': 'tool_call->result'}

Done. Each event is a future graph node; edges above are derived purely from the schema. Run the LEP-labeling demo below to populate propagation edges.


### LEP Labeling Demo → Propagation Graph

A worked example of *future* labeling: we scan the loaded trace for an obvious perturbation
(consecutive repeated tool calls = **1.3 Step Repetition**), tag those events, and — when the
run ended by exhausting its step budget (**1.5 Unaware of Termination Conditions**) — link the
repetitions as causes of that downstream failure. We then re-derive the propagation edges and
save a `*_labeled.jsonl` alongside the raw trace.

This detector is purely illustrative; the collector itself never assigns LEPs. The point is to
show the schema already supports reconstructing a cause→effect propagation graph.

In [26]:
from collections import Counter

# Work on a fresh copy loaded from disk so we don't mutate the raw `events` above.
labeled = TraceCollector.load_jsonl(PRIMARY_TRACE_PATH)

tool_calls = [e for e in labeled if e["event_type"] == "tool_call"]
target_counts = Counter(e["target"] for e in tool_calls)
repeated_targets = {t for t, c in target_counts.items() if c > 1}
final_event = next((e for e in labeled if e["event_type"] == "final_response"), None)
# metadata is no longer stored, so detect exhaustion from the fallback message itself.
exhausted = bool(final_event) and "Reached max steps" in (final_event.get("output_summary") or "")

repetition_event_ids: List[int] = []
if repeated_targets:
    # Tag each *repeat* (every occurrence after the first) of a target as Step Repetition.
    seen: set = set()
    for e in tool_calls:
        if e["target"] in repeated_targets:
            if e["target"] in seen:
                annotate_lep(labeled, e["event_id"], "1.3",
                             lep_location="agent loop", lep_severity="medium")
                repetition_event_ids.append(e["event_id"])
            seen.add(e["target"])

if exhausted and final_event is not None:
    # The run never terminated cleanly -> 1.5, flagged as a downstream failure...
    annotate_lep(labeled, final_event["event_id"], "1.5",
                 lep_location="termination", lep_severity="high",
                 downstream_failure=True, failure_type="non_termination",
                 failure_event=final_event["event_id"])
    # ...caused by the repeated steps that burned the budget.
    for rid in repetition_event_ids:
        link_propagation(labeled, rid, [final_event["event_id"]])
elif final_event is not None and not repetition_event_ids:
    # Clean run: illustrate a verification-gap label so the demo always shows something.
    annotate_lep(labeled, final_event["event_id"], "3.2",
                 lep_location="verification", lep_severity="low")

# Re-derive propagation edges now that the trace is labeled, and persist it.
labeled_edges = derive_propagation_edges(labeled)
labeled_path = PRIMARY_TRACE_PATH.with_name(PRIMARY_TRACE_PATH.stem + "_labeled.jsonl")
with labeled_path.open("w", encoding="utf-8") as fh:
    for e in labeled:
        fh.write(json.dumps(e, default=str) + "\n")

print(f"Repeated tool targets : {sorted(repeated_targets) or 'none'}")
print(f"Step-repetition events: {repetition_event_ids or 'none'}")
print(f"Run exhausted budget  : {exhausted}")
print(f"LEP propagation edges : {labeled_edges or 'none'}\n")

print("LEP-tagged events:")
for e in labeled:
    if e.get("lep_type"):
        print(f"  #{e['event_id']:>2} {e['event_type']:<14} lep={e['lep_type']!r} "
              f"cat={e['lep_category']} sev={e['lep_severity']} "
              f"caused_by={e['caused_by_event']} depends_on={e['depends_on']} "
              f"prop_to={e['propagates_to']} downstream_failure={e['downstream_failure']} "
              f"failure_type={e['failure_type']} failure_event={e['failure_event']}")
print(f"\nSaved labeled trace -> {labeled_path}")

Repeated tool targets : none
Step-repetition events: none
Run exhausted budget  : False
LEP propagation edges : none

LEP-tagged events:
  #10 final_response lep='3.2 No or Incomplete Verification' cat=FC3 sev=low caused_by=None depends_on=[] prop_to=[] downstream_failure=False failure_type=None failure_event=None

Saved labeled trace -> traces/trace_b864f75f5b08_labeled.jsonl
